In [2]:
#5.4子集树
def place(t):
    global x, a
    OK = True
    for j in range(t):
        if x[j] and a[t][j] == 0:
            OK = False
            break
    return OK

def backtrack(t):
    global cn, bestn, n, bestx, x
    if t > n:
        bestx = x[:]
        bestn = cn
        return
    if place(t - 1):
        x[t - 1] = 1
        cn += 1
        backtrack(t + 1)
        cn -= 1
    if cn + n - t > bestn:
        x[t - 1] = 0
        backtrack(t + 1)

if __name__ == "__main__":
    a = [[0, 1, 1, 0, 0],
         [1, 0, 1, 1, 1],
         [1, 1, 0, 1, 1],
         [0, 1, 1, 0, 1],
         [0, 1, 1, 1, 0]]
    n = len(a)
    x = [0 for i in range(n)]
    bestx = None
    bestn = cn = 0
    backtrack(1) # 根节点的层次为 1
    print("最大团点个数:", bestn)
    print("最大团为:", bestx)

最大团点个数: 4
最大团为: [0, 1, 1, 1, 1]


In [7]:
#5.5批处理作业调度问题——排列树
def backtrack(t):
    global f1, f2, f, x, M, bestf, bestx, n
    if t >= n:
        bestx = x[:]
        bestf = f
    else:
        for j in range(t, n):
            f1 += M[x[j]][1]
            k = t - 1
            f2[t] = max(f2[k], f1) + M[x[j]][2]
            f += f2[t]
            if f < bestf:
                x[t], x[j] = x[j], x[t]
                backtrack(t + 1)
                x[t], x[j] = x[j], x[t]
            f1 -= M[x[j]][1]
            f -= f2[t]

import sys

M = [[0, 0, 0], [0, 2, 1], [0, 3, 1], [0, 2, 3]]

f = 0
f1 = 0
n = len(M)

f2 = [i for i in range(n)]
x = [i for i in range(n)]

bestf = sys.maxsize
bestx = None

backtrack(1)

print("最优解为:", bestx)
print("最优值为:", bestf)

最优解为: [0, 1, 3, 2]
最优值为: 18


In [3]:
#6.3优先队列是分支限界法
import sys
import heapq

class Node:
    def __init__(self, cl, level, rl, x):
        self.cl = cl  # 当前路径长度
        self.level = level  # 节点的层次
        self.bl = self.cl + rl  # 路径长度下界
        self.x = x  # 部分解
    def __lt__(self, other):
        return self.bl < other.bl
    def __eq__(self, other):
        if(other == None):
            return False
        if(not isinstance(other,Node)):
            return False
        return self.bl == other.bl
def lower_bound(a):
    n = len(a)
    Minout = []
    rl = 0
    for i in range(1, n):
        minout = min(a[i])
        rl += minout
        Minout.append(minout)
    return rl, Minout
NoEdge = sys.maxsize  

def traveling(a, start, g_n):
    rl, Minout = lower_bound(a)
    heap = []
    bestx = None
    bestl = NoEdge
    node = Node(0, 2, rl, [i for i in range(g_n+1)])
    heapq.heappush(heap, node)
    while len(heap) > 0:
        current_node = heapq.heappop(heap)
        level = current_node.level
        cl = current_node.cl
        rl = current_node.bl-cl
        
        if level == g_n:
            # Update best result at leaf
            if a[current_node.x[g_n-1]][current_node.x[g_n]] != NoEdge and a[current_node.x[g_n]][1] != NoEdge and (cl + a[current_node.x[g_n-1]][current_node.x[g_n]] + a[current_node.x[g_n]][1] < bestl or bestl == NoEdge):
                bestx = current_node.x[:]
                bestl = cl + a[current_node.x[g_n-1]][current_node.x[g_n]] + a[current_node.x[g_n]][1]
        else:
            rl -= Minout[level-1]
            for j in range(level, g_n+1):
                if a[current_node.x[level-1]][current_node.x[j]] != NoEdge and (cl + rl < bestl or bestl == NoEdge):
                    current_node.x[level], current_node.x[j] = current_node.x[j], current_node.x[level]
                    heapq.heappush(heap, Node(cl + a[current_node.x[level-1]][current_node.x[level]], level + 1, rl, current_node.x[:]))
                    current_node.x[level], current_node.x[j] = current_node.x[j], current_node.x[level]
    return bestx, bestl
if __name__ == '__main__':
    a = [
        [NoEdge, NoEdge, NoEdge, NoEdge, NoEdge, NoEdge],
        [NoEdge, NoEdge, 10, NoEdge, 4, 12],
        [NoEdge, 10, NoEdge, 15, 8, 5],
        [NoEdge, NoEdge, 15, NoEdge, 7, 30],
        [NoEdge, 4, 8, 7, NoEdge, 6],
        [NoEdge, 12, 5, 30, 6, NoEdge]
    ]
    
    g_n = len(a)-1  
    bestx, bestl = traveling(a, 1, g_n)
    print("最短路径长度为:", bestl)
    print("最优旅行路线为:", bestx)

最短路径长度为: 43
最优旅行路线为: [0, 1, 4, 3, 2, 5]


In [6]:
#6.4布线问题
class Node:
    def __init__(self, row, col):
        self.row = row
        self.col = col
import queue

def findpath(start, finish):
    global grid, offset
    pathLen = 0
    if start.row == finish.row and start.col == finish.col:
        pathLen = 0
    else:
        here = start
        grid[start.row][start.col] = 0
        que = queue.Queue()
        que.put(start)
        while not que.empty():
            here = que.get()
            for i in range(4):
                nbr = Node(here.row + offset[i].row, here.col + offset[i].col)
                if grid[nbr.row][nbr.col] == -1:  
                    grid[nbr.row][nbr.col] = grid[here.row][here.col] + 1
                    if nbr.row == finish.row and nbr.col == finish.col:
                        break  
                    que.put(nbr)  
            if here.row == finish.row and here.col == finish.col:
                break  
        pathLen = grid[finish.row][finish.col]
    return pathLen
def build_path(grid, start, finish):
    global offset
    path_len = grid[finish.row][finish.col]
    path = []
    here = finish
    for j in range(path_len - 1, -1, -1):
        path.insert(0, here)
        for i in range(4):
            nbr = Node(here.row + offset[i].row, here.col + offset[i].col)
            if grid[nbr.row][nbr.col] == j:
                here = nbr
                break
    path.insert(0, start)
    return path, path_len
if __name__ == '__main__':
    n = 9 
    m = 7 

    grid = [[-1 for j in range(m + 2)] for i in range(n + 2)]

    blocked = [(1, 6), (2, 3), (3, 3), (3, 5), (3, 6), 
               (4, 3), (5, 6), (6, 5), (6, 4), (6, 6), 
               (7, 6), (9, 3), (9, 5)]
    for i, j in blocked:
        grid[i][j] = -2
    
    offset = [Node(0, 1), Node(1, 0), Node(0, -1), Node(-1, 0)]
    
    start = Node(3, 2)
    finish = Node(6, 7)
    
    for i in range(n + 2):
        grid[i][0] = -2
        grid[i][m + 1] = -2
    for j in range(m + 2):
        grid[0][j] = -2
        grid[n + 1][j] = -2
    
    findpath(start, finish)
    path, path_len = build_path(grid, start, finish)
    
    print("布线长度为:", path_len)
    print("布线方案为:")
    for i in range(len(path)):
        print(f"path[{i}].row={path[i].row} path[{i}].col={path[i].col}")

布线长度为: 10
布线方案为:
path[0].row=3 path[0].col=2
path[1].row=4 path[1].col=2
path[2].row=5 path[2].col=2
path[3].row=5 path[3].col=3
path[4].row=5 path[4].col=4
path[5].row=5 path[5].col=5
path[6].row=4 path[6].col=5
path[7].row=4 path[7].col=6
path[8].row=4 path[8].col=7
path[9].row=5 path[9].col=7
path[10].row=6 path[10].col=7
